# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

print(ds)
print(ds["train"][0])

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
        num_rows: 78835655
    })
})
{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0

In [3]:
list(ds["train"].features.keys())

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the performance of a single content page for a specific client on a specific reporting date.
I am using the fact_content_daily_performance table because it contains daily search and engagement metrics for content pages.
I will use a mid-panel month such as March 2026 (month = 2026-03) to avoid using the final outcome period.
I want to rank content pages according to their likelihood of benefiting from a content refresh.
I exclude any information that would only be known after the refresh decision because it would create data leakage.

In [4]:
print(ds["train"])

Dataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_rows: 78835655
})


In [5]:
len(ds["train"])

78835655

In [6]:
ds["train"][:3]

{'report_date': [datetime.date(2025, 1, 27),
  datetime.date(2025, 1, 27),
  datetime.date(2025, 1, 27)],
 'client_hash_id': ['client_9958f0a7ae1df715',
  'client_9958f0a7ae1df715',
  'client_9958f0a7ae1df715'],
 'content_hash_id': ['content_3b70a18ea133b2bb',
  'content_fe8e8155ce1d47a2',
  'content_b4462a1b90640058'],
 'client_has_gsc': [True, True, True],
 'client_has_ga4': [True, True, True],
 'gsc_data_available': [True, True, True],
 'ga4_data_available': [False, False, False],
 'gsc_impressions': [30, 5, 1],
 'gsc_clicks': [0, 0, 0],
 'gsc_sum_position': [115, 358, 34],
 'gsc_avg_position': [3.8333333333333335, 71.6, 34.0],
 'ga4_pageviews': [0, 0, 0],
 'ga4_sessions': [0, 0, 0],
 'ga4_users': [0, 0, 0],
 'ga4_engaged_sessions': [0, 0, 0],
 'ga4_total_engagement_sec': [0, 0, 0],
 'sessions_organic': [0, 0, 0],
 'sessions_direct': [0, 0, 0],
 'sessions_referral': [0, 0, 0],
 'sessions_social': [0, 0, 0],
 'sessions_paid': [0, 0, 0],
 'sessions_ai': [0, 0, 0],
 'ai_chatgpt': [0, 0

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Feature

The features I plan to use are:

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

## Label / Proxy

My proxy target is content refresh opportunity. Pages with weaker search performance or declining engagement may be stronger refresh candidates.

## Context

This project supports content refresh prioritization. The goal is to help decide which pages should be reviewed first.

## Excluded

I deliberately exclude information that would only be known after a refresh decision because it would create data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
ds["train"][:5]

{'report_date': [datetime.date(2025, 1, 27),
  datetime.date(2025, 1, 27),
  datetime.date(2025, 1, 27),
  datetime.date(2025, 1, 27),
  datetime.date(2025, 1, 27)],
 'client_hash_id': ['client_9958f0a7ae1df715',
  'client_9958f0a7ae1df715',
  'client_9958f0a7ae1df715',
  'client_9958f0a7ae1df715',
  'client_9958f0a7ae1df715'],
 'content_hash_id': ['content_3b70a18ea133b2bb',
  'content_fe8e8155ce1d47a2',
  'content_b4462a1b90640058',
  'content_c899aef92518c714',
  'content_c7c1d2e68d9d0964'],
 'client_has_gsc': [True, True, True, True, True],
 'client_has_ga4': [True, True, True, True, True],
 'gsc_data_available': [True, True, True, True, True],
 'ga4_data_available': [False, False, False, False, False],
 'gsc_impressions': [30, 5, 1, 6, 5],
 'gsc_clicks': [0, 0, 0, 0, 0],
 'gsc_sum_position': [115, 358, 34, 140, 89],
 'gsc_avg_position': [3.8333333333333335,
  71.6,
  34.0,
  23.333333333333332,
  17.8],
 'ga4_pageviews': [0, 0, 0, 0, 0],
 'ga4_sessions': [0, 0, 0, 0, 0],
 'ga4_use

In [8]:
print("Total Rows:", len(ds["train"]))

Total Rows: 78835655


The table contains 78,835,655 rows, indicating a large-scale warehouse dataset.

In [9]:
sample = ds["train"].select(range(10000))

available_gsc = sum(
    1 for row in sample
    if row["gsc_data_available"] is True
)

available_both = sum(
    1 for row in sample
    if row["gsc_data_available"] is True
    and row["ga4_data_available"] is True
)

print("GSC Available:", available_gsc)
print("Both Available:", available_both)

GSC Available: 10000
Both Available: 0


Availability was checked using TRUE availability flags. Only rows with available data should be considered for analysis.

In [10]:
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

print(features)

['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


### Feature 1: gsc_impressions
Knowable at the decision moment because impressions are historical search data already collected.

### Feature 2: gsc_clicks
Knowable because click data exists before the refresh decision.

### Feature 3: gsc_avg_position
Knowable because ranking information is already available from Search Console.

### Feature 4: ga4_pageviews
Knowable because pageview activity has already occurred.

### Feature 5: ga4_engaged_sessions
Knowable because engagement metrics are historical observations.

In [11]:
sample = ds["train"][:100]

target = [
    1 if x > 0 else 0
    for x in sample["ga4_pageviews"]
]

print(target[:10])

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


If I create a target directly from ga4_pageviews and also use ga4_pageviews as a feature, model performance would appear unrealistically high.

This is leakage because the feature contains information used to create the target.

The leaky feature should be removed before any real modeling.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Limitation

This dataset contains search and engagement metrics but does not directly measure content quality, user intent, or future search engine changes.

Because of this, any conclusions should be treated as directional rather than definitive.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## Self Check

✓ Defined the unit of analysis

✓ Defined the time window

✓ Defined features

✓ Defined a target/proxy

✓ Listed an excluded variable

✓ Completed three verification queries

✓ Built a five-feature frame

✓ Demonstrated leakage

✓ Named a limitation